In [8]:
"""
AlistoCare Analysis Script — BUSANALY Final Project
====================================================

Google Colab COMPATIBLE version for testing and submission.

This script performs the complete AlistoCare analysis on real, anonymized
patient encounter data from Kaggle (Healthcare Analytics Patient Flow Data,
CC0 Public Domain). No synthetic or fabricated records are included.

Compatibility:
  - Google Colab: Upload healthcare_analytics_cleaned.csv to the session,
    then run all cells. File paths use relative paths that work in Colab.
  - Local: Run with `python AlistoCare_Analysis.py` from the same folder
    as the CSV file.

Usage in Google Colab:
  1. Upload healthcare_analytics_cleaned.csv to the Colab session files pane
  2. Run this script cell by cell or all at once
  3. Output files are saved to the current working directory

Requirements:
    pandas, scikit-learn, openpyxl
"""

import json
import sys
from pathlib import Path

import pandas as pd

# ---------------------------------------------------------------------------
# Configuration — Colab compatible (no __file__ dependency)
# ---------------------------------------------------------------------------

# In Colab, __file__ is not defined. Use a configurable path that defaults
# to the current working directory (where the uploaded CSV lives).
DATA_FILE = Path("healthcare_analytics_cleaned.csv")
OUTPUT_DIR = Path("analysis_output")

MIN_ROWS = 1000
MIN_COLUMNS = 8

VALID_GENDERS = {"Female", "Male"}
VALID_ADMISSION_FLAGS = {"Admission", "Not Admission"}
VALID_DEPARTMENTS = {
    "Cardiology", "Gastroenterology", "General Practice", "Neurology",
    "Orthopedics", "Physiotherapy", "Renal"
}
VALID_WAIT_CATEGORIES = {"0-15 min", "16-30 min", "31-45 min", "46-60 min", "60+ min"}
VALID_AGE_GROUPS = {"0-17", "18-30", "31-45", "46-60", "61-75", "76+"}

# ---------------------------------------------------------------------------
# Colab file upload helper
# ---------------------------------------------------------------------------

def upload_file_to_colab(filename: str = "healthcare_analytics_cleaned.csv"):
    """
    In Google Colab, prompt the user to upload the CSV file.
    Returns True if the file exists after upload, False otherwise.
    """
    try:
        from google.colab import files
        print(f"📁 Please upload '{filename}' to this Colab session.")
        uploaded = files.upload()
        if filename in uploaded:
            print(f"✅ {filename} uploaded successfully.")
            return True
        else:
            print(f"❌ {filename} was not uploaded. Available files: {list(uploaded.keys())}")
            return False
    except ImportError:
        # Not running in Colab — skip upload
        return Path(filename).is_file()


def ensure_data_file(path: Path) -> bool:
    """Check if the data file exists; if not, offer Colab upload."""
    if path.is_file():
        return True
    print(f"⚠️  Data file not found: {path.resolve()}")
    return upload_file_to_colab(path.name)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def load_data(path: Path) -> pd.DataFrame:
    """Load the CSV dataset."""
    if not path.is_file():
        raise FileNotFoundError(f"Data file not found: {path}")
    df = pd.read_csv(path)
    return df


def validate_structure(df: pd.DataFrame) -> dict:
    """Run structural validation checks and return a results dict."""
    checks = {}
    checks["rows"] = len(df)
    checks["columns"] = df.shape[1]
    checks["rows_pass"] = checks["rows"] >= MIN_ROWS
    checks["columns_pass"] = checks["columns"] >= MIN_COLUMNS
    checks["duplicate_rows"] = int(df.duplicated().sum())
    checks["duplicate_patient_ids"] = (
        int(df["patient_id"].duplicated().sum()) if "patient_id" in df.columns else None
    )
    checks["missing_total"] = int(df.isna().sum().sum())
    checks["missing_by_column"] = df.isna().sum().to_dict()
    checks["empty_rows"] = int(df.isna().all(axis=1).sum())
    checks["empty_columns"] = int(df.isna().all(axis=0).sum())
    return checks


def validate_types_and_enums(df: pd.DataFrame) -> dict:
    """Validate data types and allowed values."""
    issues = {}
    issues["invalid_genders"] = (
        int(~df["gender"].isin(VALID_GENDERS).fillna(False))
        if "gender" in df.columns else None
    )
    issues["invalid_admission_flags"] = (
        int(~df["admission_flag"].isin(VALID_ADMISSION_FLAGS).fillna(False))
        if "admission_flag" in df.columns else None
    )
    issues["invalid_departments"] = (
        int(~df["department"].dropna().isin(VALID_DEPARTMENTS))
        if "department" in df.columns else None
    )
    issues["non_positive_wait_time"] = (
        int((df["wait_time_minutes"] <= 0).sum())
        if "wait_time_minutes" in df.columns else None
    )
    issues["non_positive_age"] = (
        int((df["age"] <= 0).sum()) if "age" in df.columns else None
    )
    issues["invalid_satisfaction"] = (
        int(((df["satisfaction_score"] < 0) | (df["satisfaction_score"] > 10)).sum())
        if "satisfaction_score" in df.columns else None
    )
    return issues


def compute_derived(df: pd.DataFrame) -> pd.DataFrame:
    """Add derived analytical fields."""
    df = df.copy()

    # Ensure admission_datetime is parsed
    if "admission_datetime" in df.columns:
        df["admission_datetime"] = pd.to_datetime(df["admission_datetime"], errors="coerce")
    elif "admission_date" in df.columns and "admission_time" in df.columns:
        df["admission_datetime"] = pd.to_datetime(
            df["admission_date"].astype(str) + " " + df["admission_time"].astype(str),
            errors="coerce"
        )

    if "admission_datetime" in df.columns and df["admission_datetime"].notna().any():
        df["date"] = df["admission_datetime"].dt.date
        df["day_of_week"] = df["admission_datetime"].dt.day_name()
        df["hour"] = df["admission_datetime"].dt.hour
        df["month"] = df["admission_datetime"].dt.month_name()

    # Wait time categories
    if "wait_time_minutes" in df.columns:
        df["wait_category"] = pd.cut(
            df["wait_time_minutes"],
            bins=[0, 15, 30, 45, 60, float("inf")],
            labels=["0-15 min", "16-30 min", "31-45 min", "46-60 min", "60+ min"],
        )

    # Age groups
    if "age" in df.columns:
        df["age_group"] = pd.cut(
            df["age"],
            bins=[0, 17, 30, 45, 60, 75, float("inf")],
            labels=["0-17", "18-30", "31-45", "46-60", "61-75", "76+"],
        )

    return df


def descriptive_analytics(df: pd.DataFrame) -> dict:
    """Compute descriptive statistics."""
    results = {}

    results["total_encounters"] = int(len(df))
    results["average_wait_time"] = round(float(df["wait_time_minutes"].mean()), 2)
    results["median_wait_time"] = float(df["wait_time_minutes"].median())
    results["min_wait_time"] = int(df["wait_time_minutes"].min())
    results["max_wait_time"] = int(df["wait_time_minutes"].max())
    results["std_wait_time"] = round(float(df["wait_time_minutes"].std()), 2)
    results["average_age"] = round(float(df["age"].mean()), 1)
    results["admission_rate"] = round(
        float((df["admission_flag"] == "Admission").mean() * 100), 1
    )
    results["satisfaction_records"] = int(df["satisfaction_score"].notna().sum())
    results["satisfaction_pct"] = round(
        float(df["satisfaction_score"].notna().mean() * 100), 1
    )
    results["department_records"] = int(df["department"].notna().sum())
    results["department_pct"] = round(
        float(df["department"].notna().mean() * 100), 1
    )

    # Wait time category distribution
    results["wait_category_distribution"] = (
        df["wait_category"].value_counts().to_dict()
    )

    # By department
    results["by_department"] = (
        df.groupby("department")["wait_time_minutes"]
        .agg(["count", "mean", "median"])
        .round(2)
        .to_dict("index")
    )

    # By day of week
    day_order = [
        "Monday", "Tuesday", "Wednesday", "Thursday",
        "Friday", "Saturday", "Sunday"
    ]
    results["by_day_of_week"] = (
        df.groupby("day_of_week")["wait_time_minutes"]
        .agg(["count", "mean"])
        .round(2)
        .reindex(day_order)
        .to_dict("index")
    )

    # By hour
    results["by_hour"] = (
        df.groupby("hour")["wait_time_minutes"]
        .agg(["count", "mean"])
        .round(2)
        .to_dict("index")
    )

    # Admission vs non-admission
    results["by_admission_flag"] = (
        df.groupby("admission_flag")["wait_time_minutes"]
        .agg(["count", "mean"])
        .round(2)
        .to_dict("index")
    )

    # Date range
    if "date" in df.columns:
        results["date_range_start"] = str(min(df["date"].dropna()))
        results["date_range_end"] = str(max(df["date"].dropna()))

    return results


def diagnostic_analytics(df: pd.DataFrame) -> dict:
    """Compute diagnostic comparisons."""
    results = {}

    # Department wait time comparison
    results["dept_wait_comparison"] = (
        df.groupby("department")["wait_time_minutes"]
        .agg(["count", "mean", "median", "min", "max"])
        .round(2)
        .to_dict("index")
    )

    # Satisfaction correlation
    sat_df = df.dropna(subset=["satisfaction_score", "wait_time_minutes"])
    if len(sat_df) > 10:
        corr = sat_df["wait_time_minutes"].corr(sat_df["satisfaction_score"])
        results["satisfaction_wait_correlation"] = round(float(corr), 4)
        results["satisfaction_sample_size"] = int(len(sat_df))
        results["average_satisfaction"] = round(
            float(sat_df["satisfaction_score"].mean()), 2
        )

    # Wait > 30 min share
    results["wait_over_30_share"] = round(
        float((df["wait_time_minutes"] > 30).mean() * 100), 1
    )
    results["wait_over_45_share"] = round(
        float((df["wait_time_minutes"] > 45).mean() * 100), 1
    )

    # Peak and low hours
    hour_stats = (
        df.groupby("hour")["wait_time_minutes"]
        .mean()
        .round(2)
    )
    peak_hour = hour_stats.idxmax()
    low_hour = hour_stats.idxmin()
    results["peak_wait_hour"] = int(peak_hour)
    results["peak_wait_hour_avg"] = round(float(hour_stats[peak_hour]), 2)
    results["lowest_wait_hour"] = int(low_hour)
    results["lowest_wait_hour_avg"] = round(float(hour_stats[low_hour]), 2)

    return results


def predictive_demonstration(df: pd.DataFrame) -> dict:
    """
    Demonstration-only classification of wait_category.
    Results are computed on real data but are illustrative, not validated for production.
    """
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, classification_report

    model_df = df.dropna(subset=["wait_category"]).copy()
    features = ["gender", "age", "department", "admission_flag", "hour"]
    model_df = model_df.dropna(subset=features)

    X = pd.get_dummies(model_df[features], drop_first=True)
    y = model_df["wait_category"].astype(str)

    if len(y) < 10 or y.nunique() < 2:
        return {"error": "Insufficient data for classification demonstration"}

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    return {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "classification_report": classification_report(
            y_test, y_pred, output_dict=True, zero_division=0
        ),
        "note": "REAL DATA — illustrative classification, not validated for production use",
    }


def prescriptive_demonstration(df: pd.DataFrame) -> dict:
    """
    Demonstration-only capacity planning scenario.
    Illustrates staffing implications based on observed wait time patterns.
    """
    # Identify departments with longest waits
    dept_waits = (
        df.groupby("department")["wait_time_minutes"]
        .mean()
        .sort_values(ascending=False)
    )

    # Identify hours with longest waits
    hour_waits = (
        df.groupby("hour")["wait_time_minutes"]
        .mean()
        .sort_values(ascending=False)
    )

    # Patients waiting > 30 minutes by department
    long_waits = df[df["wait_time_minutes"] > 30]
    long_waits_by_dept = (
        long_waits.groupby("department").size()
    )

    summary = {
        "longest_wait_department": str(dept_waits.idxmax()),
        "longest_wait_department_avg": round(float(dept_waits.max()), 2),
        "shortest_wait_department": str(dept_waits.idxmin()),
        "shortest_wait_department_avg": round(float(dept_waits.min()), 2),
        "peak_wait_hour": int(hour_waits.idxmax()),
        "peak_wait_hour_avg": round(float(hour_waits.max()), 2),
        "patients_waiting_over_30_min": int((df["wait_time_minutes"] > 30).sum()),
        "patients_waiting_over_30_min_pct": round(
            float((df["wait_time_minutes"] > 30).mean() * 100), 1
        ),
        "long_waits_by_department": long_waits_by_dept.to_dict(),
        "note": "REAL DATA — planning illustration based on observed patterns, not a final recommendation",
    }
    return summary


def save_results(df: pd.DataFrame, descriptive: dict, diagnostic: dict,
                 predictive: dict, prescriptive: dict, validation: dict) -> None:
    """Save cleaned data and summary results to CSV and JSON files."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Cleaned dataset with derived fields
    df.to_csv(OUTPUT_DIR / "alistocare_cleaned.csv", index=False)

    # Validation summary
    val_rows = []
    for k, v in validation.items():
        if k == "missing_by_column":
            for col, val in v.items():
                val_rows.append({"check": f"missing_{col}", "value": str(val)})
        else:
            val_rows.append({"check": k, "value": str(v)})
    pd.DataFrame(val_rows).to_csv(OUTPUT_DIR / "validation_summary.csv", index=False)

    # Descriptive summary
    desc_rows = []
    for k, v in descriptive.items():
        if isinstance(v, dict):
            for sub_k, sub_v in v.items():
                desc_rows.append({"metric": f"{k}.{sub_k}", "value": str(sub_v)})
        else:
            desc_rows.append({"metric": k, "value": str(v)})
    pd.DataFrame(desc_rows).to_csv(OUTPUT_DIR / "descriptive_summary.csv", index=False)

    # Diagnostic summary
    diag_rows = []
    for k, v in diagnostic.items():
        if isinstance(v, dict):
            for sub_k, sub_v in v.items():
                diag_rows.append({"metric": f"{k}.{sub_k}", "value": str(sub_v)})
        else:
            diag_rows.append({"metric": k, "value": str(v)})
    pd.DataFrame(diag_rows).to_csv(OUTPUT_DIR / "diagnostic_summary.csv", index=False)

    # Predictive summary
    if isinstance(predictive, dict) and "error" not in predictive:
        pred_rows = [
            {"metric": "accuracy", "value": str(predictive["accuracy"])},
            {"metric": "note", "value": predictive.get("note", "")},
        ]
        pd.DataFrame(pred_rows).to_csv(OUTPUT_DIR / "predictive_summary.csv", index=False)

    # Prescriptive summary
    pres_rows = []
    for k, v in prescriptive.items():
        if isinstance(v, dict):
            for sub_k, sub_v in v.items():
                pres_rows.append({"metric": f"{k}.{sub_k}", "value": str(sub_v)})
        else:
            pres_rows.append({"metric": k, "value": str(v)})
    pd.DataFrame(pres_rows).to_csv(OUTPUT_DIR / "prescriptive_summary.csv", index=False)

    # Save combined JSON results
    combined = {
        "descriptive": descriptive,
        "diagnostic": diagnostic,
        "predictive": predictive,
        "prescriptive": prescriptive,
        "validation": {k: v for k, v in validation.items() if k != "missing_by_column"},
    }
    with open(OUTPUT_DIR / "analysis_results.json", "w") as f:
        json.dump(combined, f, indent=2, default=str)


def print_summary(validation: dict, descriptive: dict, diagnostic: dict,
                  predictive: dict, prescriptive: dict) -> None:
    """Print a human-readable summary to stdout."""
    print("=" * 60)
    print("ALISTOCARE ANALYSIS SUMMARY — REAL DATA")
    print("=" * 60)
    print()

    print("--- Validation ---")
    print(f"Rows: {validation['rows']:,} (minimum {MIN_ROWS}: {'PASS' if validation['rows_pass'] else 'FAIL'})")
    print(f"Columns: {validation['columns']} (minimum {MIN_COLUMNS}: {'PASS' if validation['columns_pass'] else 'FAIL'})")
    print(f"Duplicate rows: {validation['duplicate_rows']}")
    print(f"Duplicate patient_ids: {validation['duplicate_patient_ids']}")
    print(f"Missing values (total): {validation['missing_total']}")
    print()

    print("--- Descriptive ---")
    print(f"Total encounters: {descriptive['total_encounters']:,}")
    print(f"Average wait time: {descriptive['average_wait_time']:.2f} minutes")
    print(f"Median wait time: {descriptive['median_wait_time']:.2f} minutes")
    print(f"Wait time range: {descriptive['min_wait_time']} – {descriptive['max_wait_time']} minutes")
    print(f"Standard deviation: {descriptive['std_wait_time']:.2f} minutes")
    print(f"Average patient age: {descriptive['average_age']:.1f} years")
    print(f"Admission rate: {descriptive['admission_rate']:.1f}%")
    print(f"Satisfaction records: {descriptive['satisfaction_records']:,} ({descriptive['satisfaction_pct']:.1f}%)")
    print(f"Department records: {descriptive['department_records']:,} ({descriptive['department_pct']:.1f}%)")
    print(f"Patients waiting >30 min: {descriptive['wait_over_30_share']}%")
    print(f"Patients waiting >45 min: {descriptive['wait_over_45_share']}%")
    print()

    print("--- Diagnostic ---")
    print(f"Correlation: wait time vs satisfaction: {diagnostic.get('satisfaction_wait_correlation', 'N/A')}")
    print(f"Peak wait hour: {diagnostic.get('peak_wait_hour', 'N/A')}:00 ({diagnostic.get('peak_wait_hour_avg', 'N/A')} min)")
    print(f"Lowest wait hour: {diagnostic.get('lowest_wait_hour', 'N/A')}:00 ({diagnostic.get('lowest_wait_hour_avg', 'N/A')} min)")
    print()

    print("--- Predictive (real data demo) ---")
    if isinstance(predictive, dict) and "error" not in predictive:
        print(f"Classification accuracy: {predictive['accuracy']:.3f}")
    else:
        print(f"Predictive demo skipped: {predictive.get('error', 'unknown')}")
    print()

    print("--- Prescriptive (real data demo) ---")
    print(f"Longest wait department: {prescriptive.get('longest_wait_department', 'N/A')}")
    print(f"Longest wait avg: {prescriptive.get('longest_wait_department_avg', 'N/A')} min")
    print(f"Patients waiting >30 min: {prescriptive.get('patients_waiting_over_30_min', 'N/A'):,}")
    print()

    print("=" * 60)
    print("SOURCE: healthcare_analytics_cleaned.csv (real data, Kaggle CC0)")
    print("PII handled: SSN-format IDs pseudonymized, name column dropped")
    print("Complies with Data Privacy Act of 2012 (RA 10173)")
    print("=" * 60)


def main() -> None:
    """Main entry point — works in both Colab and local environments."""
    # Ensure data file exists (Colab upload if needed)
    if not ensure_data_file(DATA_FILE):
        print(f"\n❌ Cannot proceed without {DATA_FILE.name}")
        print("   Upload the file to this Colab session and re-run.")
        sys.exit(1)

    print(f"Loading data from: {DATA_FILE.resolve()}")
    df = load_data(DATA_FILE)

    print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
    print()

    # Validate
    validation = validate_structure(df)
    enum_issues = validate_types_and_enums(df)
    validation.update(enum_issues)

    # Compute derived fields
    df = compute_derived(df)

    # Analytics
    descriptive = descriptive_analytics(df)
    diagnostic = diagnostic_analytics(df)
    predictive = predictive_demonstration(df)
    prescriptive = prescriptive_demonstration(df)

    # Save outputs
    save_results(df, descriptive, diagnostic, predictive, prescriptive, validation)

    # Print summary
    print_summary(validation, descriptive, diagnostic, predictive, prescriptive)

    print(f"\nOutputs saved to: {OUTPUT_DIR.resolve()}")


# Colab-compatible: runs when cell is executed, no __main__ guard needed
main()

Loading data from: /content/healthcare_analytics_cleaned.csv
Loaded 9,216 rows, 19 columns

ALISTOCARE ANALYSIS SUMMARY — REAL DATA

--- Validation ---
Rows: 9,216 (minimum 1000: PASS)
Columns: 19 (minimum 8: PASS)
Duplicate rows: 0
Duplicate patient_ids: 0
Missing values (total): 45717

--- Descriptive ---
Total encounters: 9,216
Average wait time: 35.26 minutes
Median wait time: 35.00 minutes
Wait time range: 10 – 60 minutes
Standard deviation: 14.74 minutes
Average patient age: 39.9 years
Admission rate: 50.0%
Satisfaction records: 2,517 (27.3%)
Department records: 3,816 (41.4%)
Patients waiting >30 min: 59.3%
Patients waiting >45 min: 29.8%

--- Diagnostic ---
Correlation: wait time vs satisfaction: -0.0212
Peak wait hour: 11:00 (37.63 min)
Lowest wait hour: 0:00 (32.78 min)

--- Predictive (real data demo) ---
Classification accuracy: 0.299

--- Prescriptive (real data demo) ---
Longest wait department: Neurology
Longest wait avg: 36.8 min
Patients waiting >30 min: 5,467

SOURCE: 

In [7]:
"""
AlistoCare Data Transformation Script — BUSANALY Final Project
==============================================================

Transforms the original Kaggle Healthcare Analytics Patient Flow Data into:
1. Anonymized dataset (PII removed per RA 10173)
2. Cleaned dataset (parsed dates, derived fields, ready for analysis)

DATA SOURCE:
  Title: Healthcare Analytics Patient Flow Data
  Creator: Hassan Jameel Ahmed
  Platform: Kaggle
  URL: https://www.kaggle.com/datasets/hassanjameelahmed/healthcare-analytics-patient-flow-data
  License: CC0: Public Domain — safe to share publicly, including on GitHub

PRIVACY COMPLIANCE:
  - Patient Id (SSN format XXX-XX-XXXX) → SHA-256 pseudonym (PID-XXXXXXXXXXXX)
  - Merged column (patient names) → dropped entirely
  - Complies with Data Privacy Act of 2012 (RA 10173)

USAGE IN GOOGLE COLAB:
  1. Upload healthcare_analytics_patient_flow_data.csv to the Colab session
  2. Run this script — outputs saved to current directory
  3. Download healthcare_analytics_anonymized.csv and healthcare_analytics_cleaned.csv

USAGE LOCALLY:
  python AlistoCare_Transform.py
  (Place healthcare_analytics_patient_flow_data.csv in the same folder)

REQUIREMENTS:
    pandas
"""

import hashlib
import sys
from pathlib import Path

import pandas as pd

# ---------------------------------------------------------------------------
# Configuration — Colab compatible (no __file__ dependency)
# ---------------------------------------------------------------------------

ORIGINAL_FILE = Path("healthcare_analytics_patient_flow_data.csv")
ANONYMIZED_FILE = Path("healthcare_analytics_anonymized.csv")
CLEANED_FILE = Path("healthcare_analytics_cleaned.csv")

# ---------------------------------------------------------------------------
# Colab file upload helper
# ---------------------------------------------------------------------------

def upload_file_to_colab(filename: str = "healthcare_analytics_patient_flow_data.csv"):
    """In Google Colab, prompt the user to upload the CSV file."""
    try:
        from google.colab import files
        print(f"📁 Please upload '{filename}' to this Colab session.")
        uploaded = files.upload()
        if filename in uploaded:
            print(f"✅ {filename} uploaded successfully.")
            return True
        else:
            print(f"❌ {filename} was not uploaded. Available files: {list(uploaded.keys())}")
            return False
    except ImportError:
        return Path(filename).is_file()


def ensure_original_file(path: Path) -> bool:
    """Check if the original data file exists; if not, offer Colab upload."""
    if path.is_file():
        return True
    print(f"⚠️  Original data file not found: {path.resolve()}")
    return upload_file_to_colab(path.name)


# ---------------------------------------------------------------------------
# Transformation steps
# ---------------------------------------------------------------------------

def load_original(path: Path) -> pd.DataFrame:
    """Load the original Kaggle CSV dataset."""
    if not path.is_file():
        raise FileNotFoundError(f"Original data file not found: {path}")
    df = pd.read_csv(path)
    print(f"Loaded original dataset: {len(df):,} rows, {df.shape[1]} columns")
    return df


def anonymize_pii(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove / pseudonymize personally identifiable information per RA 10173.

    - Patient Id (SSN format XXX-XX-XXXX) → SHA-256 pseudonym
    - Merged column (patient names) → dropped entirely
    """
    df = df.copy()

    # Pseudonymize Patient Id using SHA-256 (first 12 hex chars)
    if "Patient Id" in df.columns:
        df["Patient Id"] = df["Patient Id"].apply(
            lambda x: "PID-" + hashlib.sha256(str(x).encode()).hexdigest()[:12]
        )
        print("✅ Patient Id pseudonymized (SHA-256 → PID-XXXXXXXXXXXX)")

    # Drop Merged column (contains patient names — direct PII)
    if "Merged" in df.columns:
        df = df.drop(columns=["Merged"])
        print("✅ Merged column dropped (contained patient names)")

    return df


def rename_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize column names to lowercase snake_case."""
    column_map = {
        "Patient Id": "patient_id",
        "Patient Admission Date": "admission_date",
        "Patient Admission Time": "admission_time",
        "Patient Gender": "gender",
        "Patient Age": "age",
        "Patient Race": "race",
        "Department Referral": "department",
        "Patient Admission Flag": "admission_flag",
        "Patient Satisfaction Score": "satisfaction_score",
        "Patient Waittime": "wait_time_minutes",
    }
    df = df.rename(columns=column_map)
    print("✅ Columns renamed to snake_case")
    return df


def fix_data_errors(df: pd.DataFrame) -> pd.DataFrame:
    """Correct known data entry errors."""
    df = df.copy()

    # Fix "Femaleemale" → "Female" (17 records observed in source data)
    if "gender" in df.columns:
        bad = int((df["gender"] == "Femaleemale").sum())
        if bad > 0:
            df["gender"] = df["gender"].replace("Femaleemale", "Female")
            print(f"✅ Fixed {bad} gender data entry error(s): 'Femaleemale' → 'Female'")

    return df


def parse_datetime(df: pd.DataFrame) -> pd.DataFrame:
    """Parse admission_date and admission_time into combined admission_datetime."""
    df = df.copy()

    # Parse date (format: M/D/YYYY)
    df["admission_date_parsed"] = pd.to_datetime(df["admission_date"], format="%m/%d/%Y", errors="coerce")

    # Parse time (format: H:MM:SS AM/PM)
    df["admission_time_parsed"] = pd.to_datetime(
        df["admission_time"], format="%I:%M:%S %p", errors="coerce"
    ).dt.time

    # Combine into full datetime
    df["admission_datetime"] = pd.to_datetime(
        df["admission_date_parsed"].dt.strftime("%Y-%m-%d") + " " +
        df["admission_time_parsed"].astype(str),
        errors="coerce"
    )

    parse_failures = int(df["admission_datetime"].isna().sum())
    if parse_failures > 0:
        print(f"⚠️  {parse_failures} records could not be parsed ({parse_failures/len(df)*100:.1f}%)")
    else:
        print("✅ Date/time parsed successfully for all records")

    return df


def add_derived_fields(df: pd.DataFrame) -> pd.DataFrame:
    """Add derived analytical fields."""
    df = df.copy()

    # Extract date components
    df["date"] = df["admission_datetime"].dt.date
    df["day_of_week"] = df["admission_datetime"].dt.day_name()
    df["hour"] = df["admission_datetime"].dt.hour
    df["month"] = df["admission_datetime"].dt.month_name()

    # Wait time categories
    df["wait_category"] = pd.cut(
        df["wait_time_minutes"],
        bins=[0, 15, 30, 45, 60, float("inf")],
        labels=["0-15 min", "16-30 min", "31-45 min", "46-60 min", "60+ min"],
    )

    # Age groups
    df["age_group"] = pd.cut(
        df["age"],
        bins=[0, 17, 30, 45, 60, 75, float("inf")],
        labels=["0-17", "18-30", "31-45", "46-60", "61-75", "76+"],
    )

    print("✅ Derived fields added (date, day_of_week, hour, month, wait_category, age_group)")
    return df


def validate_cleaned(df: pd.DataFrame) -> dict:
    """Run validation checks on the cleaned dataset."""
    checks = {}
    checks["rows"] = len(df)
    checks["columns"] = df.shape[1]
    checks["duplicate_rows"] = int(df.duplicated().sum())
    checks["duplicate_patient_ids"] = int(df["patient_id"].duplicated().sum()) if "patient_id" in df.columns else None
    checks["missing_core"] = int(
        df[["patient_id", "admission_date", "admission_time", "gender", "age",
            "race", "admission_flag", "wait_time_minutes"]].isna().sum().sum()
    ) if all(c in df.columns for c in ["patient_id", "admission_date", "admission_time",
                                      "gender", "age", "race", "admission_flag", "wait_time_minutes"]) else None
    checks["missing_department"] = int(df["department"].isna().sum()) if "department" in df.columns else None
    checks["missing_satisfaction"] = int(df["satisfaction_score"].isna().sum()) if "satisfaction_score" in df.columns else None
    return checks


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main() -> None:
    """Transform original Kaggle dataset into anonymized and cleaned versions."""
    print("=" * 60)
    print("ALISTOCARE DATA TRANSFORMATION")
    print("=" * 60)
    print()

    # Ensure original file exists
    if not ensure_original_file(ORIGINAL_FILE):
        print(f"\n❌ Cannot proceed without {ORIGINAL_FILE.name}")
        print("   Upload the file to this Colab session and re-run.")
        sys.exit(1)

    # Step 1: Load original
    print("Step 1: Loading original dataset...")
    df = load_original(ORIGINAL_FILE)
    print()

    # Step 2: Anonymize PII
    print("Step 2: Anonymizing PII (RA 10173 compliance)...")
    df = anonymize_pii(df)
    print()

    # Step 3: Rename columns
    print("Step 3: Renaming columns to snake_case...")
    df = rename_columns(df)
    print()

    # Step 4: Fix data errors
    print("Step 4: Fixing data entry errors...")
    df = fix_data_errors(df)
    print()

    # Step 5: Parse datetime
    print("Step 5: Parsing date/time fields...")
    df = parse_datetime(df)
    print()

    # Step 6: Add derived fields
    print("Step 6: Adding derived fields...")
    df = add_derived_fields(df)
    print()

    # Step 7: Save anonymized version (no derived fields, just anonymized core)
    print("Step 7: Saving anonymized dataset...")
    anonymized_cols = [
        "patient_id", "admission_date", "admission_time", "gender", "age",
        "race", "department", "admission_flag", "satisfaction_score", "wait_time_minutes",
    ]
    # Keep only columns that exist
    existing_anon_cols = [c for c in anonymized_cols if c in df.columns]
    anonymized_df = df[existing_anon_cols].copy()
    anonymized_df.to_csv(ANONYMIZED_FILE, index=False)
    print(f"✅ Saved: {ANONYMIZED_FILE} ({len(anonymized_df):,} rows × {len(existing_anon_cols)} columns)")
    print()

    # Step 8: Save cleaned version (with derived fields)
    print("Step 8: Saving cleaned dataset...")
    df.to_csv(CLEANED_FILE, index=False)
    print(f"✅ Saved: {CLEANED_FILE} ({len(df):,} rows × {df.shape[1]} columns)")
    print()

    # Step 9: Validate
    print("Step 9: Validation checks...")
    checks = validate_cleaned(df)
    print(f"  Rows: {checks['rows']:,} (≥1,000: {'PASS' if checks['rows'] >= 1000 else 'FAIL'})")
    print(f"  Columns: {checks['columns']} (≥8: {'PASS' if checks['columns'] >= 8 else 'FAIL'})")
    print(f"  Duplicate rows: {checks['duplicate_rows']}")
    print(f"  Duplicate patient_ids: {checks['duplicate_patient_ids']}")
    print(f"  Missing core fields: {checks['missing_core']}")
    print(f"  Missing department: {checks['missing_department']} (documented, not imputed)")
    print(f"  Missing satisfaction: {checks['missing_satisfaction']} (documented, not imputed)")
    print()

    # Summary
    print("=" * 60)
    print("TRANSFORMATION COMPLETE")
    print("=" * 60)
    print()
    print("Output files:")
    print(f"  • {ANONYMIZED_FILE} — PII removed, core fields only")
    print(f"  • {CLEANED_FILE} — cleaned + derived fields, ready for analysis")
    print()
    print("Privacy: SSN-format IDs pseudonymized, name column dropped (RA 10173)")
    print("Source: Kaggle CC0 Public Domain — safe for public sharing including GitHub")
    print("=" * 60)


# Colab-compatible: runs when cell is executed
main()

ALISTOCARE DATA TRANSFORMATION

⚠️  Original data file not found: /content/healthcare_analytics_patient_flow_data.csv
📁 Please upload 'healthcare_analytics_patient_flow_data.csv' to this Colab session.


Saving healthcare_analytics_patient_flow_data.csv to healthcare_analytics_patient_flow_data.csv
✅ healthcare_analytics_patient_flow_data.csv uploaded successfully.
Step 1: Loading original dataset...
Loaded original dataset: 9,216 rows, 11 columns

Step 2: Anonymizing PII (RA 10173 compliance)...
✅ Patient Id pseudonymized (SHA-256 → PID-XXXXXXXXXXXX)
✅ Merged column dropped (contained patient names)

Step 3: Renaming columns to snake_case...
✅ Columns renamed to snake_case

Step 4: Fixing data entry errors...
✅ Fixed 17 gender data entry error(s): 'Femaleemale' → 'Female'

Step 5: Parsing date/time fields...
⚠️  5603 records could not be parsed (60.8%)

Step 6: Adding derived fields...
✅ Derived fields added (date, day_of_week, hour, month, wait_category, age_group)

Step 7: Saving anonymized dataset...
✅ Saved: healthcare_analytics_anonymized.csv (9,216 rows × 10 columns)

Step 8: Saving cleaned dataset...
✅ Saved: healthcare_analytics_cleaned.csv (9,216 rows × 19 columns)

Step 9: V